# Initialize `oil_network_data_loader` schema

EIA crude-oil ingestion pipeline. Three-table pattern:

- **`eia_staging`** — wide staging table; one row per EIA fetch, tagged by `source_dataset` / `route` / `frequency`.
- **`ref_timeseries`** — denormalised catalogue, one row per unique `(source, timeseries_id)`. PK = `(source, timeseries_id)`.
- **`timeseries`** — facts; PK = `(source, timeseries_id, timeseries_date, timeseries_published_date)` so re-fetches with new vintages append rather than overwrite. FK to `ref_timeseries`.

**Repeatable** by default — Step 1 drops & recreates the schema. Set `RESET_SCHEMA = False` in init to keep accumulating across runs (the timeseries PK is vintage-aware so re-fetches are safe).

**Scope:** does not touch `oil_network` or `oil_network_metadata`. Replaces the deprecated `source_eia` schema (drop manually with `DROP SCHEMA source_eia CASCADE;`).

**Loads** all 27 EIA crude datasets across 7 domains (production, imports, exports, inter-PADD movements, refinery, stocks, S&D, weekly trade). All endpoints empirically verified to return data (~145,800 rows from 2015-01-01).

Production-side coverage spans two EIA publications:
- `production` — `petroleum/crd/crpdn` — state + PADD + national crude (84 series).
- `production_steo` — `steo` — basin-level crude (Permian, Bakken, Eagle Ford, Appalachia, Haynesville) plus Federal GoM, Alaska, and Lower 48 excl GoM (8 series). STEO is the canonical EIA monthly publication for basin-level data; `crpdn` does not provide it.

## 0. Init

Imports, env-var-driven Postgres connection, EIA API config, the dataset list, and the run-mode flags.

In [1]:
import os
import re
import time
import unicodedata
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display
from sqlalchemy import create_engine, text

# Load .env from repo root (clean/) so EIA_API_KEY and PG_* are available.
load_dotenv(Path(__file__).resolve().parent.parent / ".env" if "__file__" in globals()
            else Path.cwd().parent / ".env")

# ── EIA API ────────────────────────────────────────────────────────────
API_KEY   = os.environ.get("EIA_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "EIA_API_KEY environment variable not set. "
        "Get a free key at https://www.eia.gov/opendata/ and add it to .env "
        "at the repository root, then restart the kernel."
    )
BASE_URL  = "https://api.eia.gov/v2"
START     = "2015-01-01"
PAGE_SIZE = 5000

# ── PostgreSQL ─────────────────────────────────────────────────────────
PG_HOST   = os.environ.get("PG_HOST",   "localhost")
PG_PORT   = os.environ.get("PG_PORT",   "5432")
PG_DB     = os.environ.get("PG_DB",     "eia_crude")
PG_USER   = os.environ.get("PG_USER",   "eia_user")
PG_PASS   = os.environ.get("PG_PASS",   "eia_password")
PG_SCHEMA = os.environ.get("PG_SCHEMA", "oil_network_data_loader")
PG_URL    = f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"

engine = create_engine(
    PG_URL,
    connect_args={"options": f"-csearch_path={PG_SCHEMA},public"},
    future=True,
)

# ── Run-mode flags ─────────────────────────────────────────────────────
RESET_SCHEMA  = True    # Step 1: drop + recreate the schema
RESET_STAGING = True    # Step 4: truncate eia_staging before loading

# ── Tables ─────────────────────────────────────────────────────────────
STAGING_TABLE        = "eia_staging"
REF_TIMESERIES_TABLE = "ref_timeseries"
TIMESERIES_TABLE     = "timeseries"
TIMESERIES_FK_NAME   = "timeseries_ref_timeseries_fk"

# Columns kept in eia_staging. Every fetched DataFrame is reindexed to
# this set before append, so the staging schema stays stable across
# datasets even when the EIA endpoint omits some fields.
STAGING_COLS = [
    "source_dataset", "route", "frequency", "date",
    "duoarea", "area-name",
    "product", "product-name",
    "process", "process-name",
    "series", "series-description",
    "value", "units",
]

# 27 EIA crude datasets across 7 domains.
# Naming convention for source_dataset:
#   {domain}             — core variable (production, stocks)
#   {domain}_{qualifier} — variant or breakdown (imports_padd, production_steo)
EIA_DATASETS = [
    # Production — state / PADD / national (petroleum/crd/crpdn)
    {"name": "production",           "route": "petroleum/crd/crpdn",   "frequency": "monthly", "facets": None},
    # Production — STEO basin-level (Permian, Bakken, Eagle Ford, Appalachia, Haynesville)
    # plus Federal GoM, Alaska, and Lower 48 excl GoM. STEO is the EIA monthly publication
    # that provides the basin breakdown not published by crpdn.
    {"name": "production_steo",      "route": "steo",                  "frequency": "monthly",
     "facets": {"seriesId": ["COPRPM", "COPRBK", "COPREF", "COPRAP", "COPRHA",
                             "PAPRPGLF", "PAPRPAK", "PAPR48NGOM"]}},
    # Imports
    {"name": "imports",              "route": "petroleum/move/impcus", "frequency": "monthly", "facets": {"product": "EPC0"}},
    {"name": "imports_padd",         "route": "petroleum/move/impcp",  "frequency": "monthly", "facets": {"product": "EPC0"}},
    {"name": "imports_entry",        "route": "petroleum/move/imp",    "frequency": "monthly", "facets": {"product": "EPC0"}},
    {"name": "imports_processing",   "route": "petroleum/move/imp2",   "frequency": "monthly", "facets": {"product": "EPC0"}},
    {"name": "net_imports",          "route": "petroleum/move/neti",   "frequency": "monthly", "facets": {"product": "EPC0"}},
    # Exports
    {"name": "exports",              "route": "petroleum/move/expc",   "frequency": "monthly", "facets": {"product": "EPC0"}},
    {"name": "exports_padd",         "route": "petroleum/move/expcp",  "frequency": "monthly", "facets": {"product": "EPC0"}},
    {"name": "exports_total",        "route": "petroleum/move/exp",    "frequency": "monthly", "facets": {"product": "EPC0"}},
    # Inter-PADD movements
    {"name": "movements_all",        "route": "petroleum/move/ptb",    "frequency": "monthly", "facets": {"product": "EPC0"}},
    {"name": "movements_pipeline",   "route": "petroleum/move/pipe",   "frequency": "monthly", "facets": {"product": "EPC0"}},
    {"name": "movements_tanker",     "route": "petroleum/move/tb",     "frequency": "monthly", "facets": {"product": "EPC0"}},
    {"name": "movements_rail",       "route": "petroleum/move/rail",   "frequency": "monthly", "facets": {"product": "EPC0"}},
    {"name": "movements_net",        "route": "petroleum/move/netr",   "frequency": "monthly", "facets": {"product": "EPC0"}},
    # Refinery — district-level (monthly)
    {"name": "refinery_input",       "route": "petroleum/pnp/inpt2",   "frequency": "monthly", "facets": {"product": "EPC0"}},
    {"name": "refinery_input_blend", "route": "petroleum/pnp/inpt",    "frequency": "monthly", "facets": {"product": "EPC0"}},
    {"name": "refinery_unc",         "route": "petroleum/pnp/unc",     "frequency": "monthly", "facets": None},
    {"name": "refinery_quality",     "route": "petroleum/pnp/crq",     "frequency": "monthly", "facets": None},
    # Refinery — weekly (PADD-level)
    {"name": "refinery_input_wk",    "route": "petroleum/sum/sndw",    "frequency": "weekly",  "facets": {"product": "EPC0", "process": "YIY"}},
    {"name": "refinery_wiup",        "route": "petroleum/pnp/wiup",    "frequency": "weekly",  "facets": None},
    # Stocks
    {"name": "stocks",               "route": "petroleum/stoc/cu",     "frequency": "monthly", "facets": None},
    {"name": "stocks_refinery",      "route": "petroleum/stoc/ref",    "frequency": "monthly", "facets": {"product": "EPC0"}},
    {"name": "stocks_weekly",        "route": "petroleum/stoc/wstk",   "frequency": "weekly",  "facets": {"product": "EPC0"}},
    # Supply & Disposition (national balance)
    {"name": "crude_snd",            "route": "petroleum/sum/crdsnd",  "frequency": "monthly", "facets": None},
    {"name": "supply_disposition",   "route": "petroleum/sum/snd",     "frequency": "monthly", "facets": {"product": "EPC0"}},
    # Weekly trade
    {"name": "trade_weekly",         "route": "petroleum/move/wkly",   "frequency": "weekly",  "facets": {"product": "EPC0"}},
]

def debug(msg):
    print(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {msg}")

with engine.connect() as conn:
    ver, db, usr = conn.execute(text("SELECT version(), current_database(), current_user")).one()

debug(f"Connected:        {db} as {usr}")
debug(f"Server:           {ver.splitlines()[0]}")
debug(f"Target schema:    {PG_SCHEMA}")
debug(f"EIA Key:          {API_KEY[:6]}...{API_KEY[-4:]}  (start={START})")
debug(f"Datasets:         {len(EIA_DATASETS)} configured")
debug(f"RESET_SCHEMA:     {RESET_SCHEMA}")
debug(f"RESET_STAGING:    {RESET_STAGING}")

[2026-05-18 17:17:56] Connected:        eia_crude as eia_user
[2026-05-18 17:17:56] Server:           PostgreSQL 18.3 on x86_64-windows, compiled by msvc-19.44.35223, 64-bit
[2026-05-18 17:17:56] Target schema:    oil_network_data_loader
[2026-05-18 17:17:56] EIA Key:          wPmxxY...2And  (start=2015-01-01)
[2026-05-18 17:17:56] Datasets:         27 configured
[2026-05-18 17:17:56] RESET_SCHEMA:     True
[2026-05-18 17:17:56] RESET_STAGING:    True


## 1. Reset schema

Drop and recreate `oil_network_data_loader` so the notebook is idempotent. Skip the drop (`RESET_SCHEMA = False`) when you want to accumulate vintages across runs — the timeseries PK includes `timeseries_published_date` so re-fetches land as new rows rather than overwrites.

In [2]:
with engine.begin() as conn:
    if RESET_SCHEMA:
        conn.execute(text(f'DROP SCHEMA IF EXISTS "{PG_SCHEMA}" CASCADE'))
        debug(f"  ✗ dropped {PG_SCHEMA}")
    conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{PG_SCHEMA}"'))
    debug(f"  ✓ schema {PG_SCHEMA} ready")

[2026-05-18 17:17:56]   ✗ dropped oil_network_data_loader
[2026-05-18 17:17:56]   ✓ schema oil_network_data_loader ready


## 2. Functions

Helpers used downstream:

- `fetch_eia(route, facets, frequency)` — paginated EIA v2 GET → cleaned DataFrame. Renames the API's `period` field to `date` (parsed as datetime) and coerces `value` to numeric.
- `stage_dataframe(df, name, route, freq)` — append a fetched DataFrame to `eia_staging`, reindexed to `STAGING_COLS`. Missing columns filled with NULL.
- `ensure_staging_table()` — create `eia_staging` if missing.
- `truncate_staging()` — empty `eia_staging` (keeps schema).
- `ensure_ref_tables(regenerate=False)` — create `ref_timeseries`. `regenerate=True` drops and recreates.
- `refresh_ref_timeseries_from(table)` — UPSERT distinct series metadata from `table` into `ref_timeseries`.

In [3]:
def fetch_eia(route, facets=None, frequency="monthly", extra_params=None):
    """Paginated EIA v2 fetch -> cleaned DataFrame.

    Renames the EIA response field `period` to `date` (parsed as datetime)
    before returning. The HTTP request still uses EIA's own `period` name
    as a URL param — that's their field, not ours.

    Petroleum endpoints expose `series` / `series-description` / `units`.
    STEO uses camelCase + singular: `seriesId` / `seriesDescription` / `unit`.
    Normalise to the petroleum schema so downstream staging is uniform.
    """
    url = f"{BASE_URL}/{route}/data/"
    params = {
        "api_key":            API_KEY,
        "frequency":          frequency,
        "data[0]":            "value",
        "start":              START,
        "length":             PAGE_SIZE,
        "sort[0][column]":    "period",
        "sort[0][direction]": "asc",
    }
    if facets:
        for k, v in facets.items():
            if isinstance(v, list):
                for j, item in enumerate(v):
                    params[f"facets[{k}][{j}]"] = item
            else:
                params[f"facets[{k}][]"] = v
    if extra_params:
        params.update(extra_params)

    all_records, offset, first = [], 0, True
    while True:
        params["offset"] = offset
        resp = requests.get(url, params=params, timeout=30)
        if resp.status_code != 200:
            debug(f"  ✗ HTTP {resp.status_code}: {resp.text[:200]}")
            break
        body    = resp.json()
        records = body.get("response", {}).get("data", [])
        total   = int(body.get("response", {}).get("total", 0))
        if first and records:
            debug(f"    columns: {list(records[0].keys())}")
            debug(f"    total  : {total:,}")
            first = False
        all_records.extend(records)
        offset += len(records)
        if offset >= total or not records:
            break
        time.sleep(0.15)

    if not all_records:
        return pd.DataFrame()

    df = pd.DataFrame(all_records)
    rename_map = {
        "period":            "date",
        "seriesId":          "series",
        "seriesDescription": "series-description",
    }
    if "unit" in df.columns and "units" not in df.columns:
        rename_map["unit"] = "units"
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    if "value" in df.columns:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df.sort_values("date").reset_index(drop=True)


def stage_dataframe(df, dataset_name, route, frequency):
    """Append a fetched DataFrame to STAGING_TABLE, reindexed to STAGING_COLS."""
    if df.empty:
        debug(f"  ⚠ {dataset_name}: empty DataFrame, skipped")
        return 0
    df = df.copy()
    df["source_dataset"] = dataset_name
    df["route"]          = route
    df["frequency"]      = frequency
    df = df.reindex(columns=STAGING_COLS)
    df.to_sql(STAGING_TABLE, engine, schema=PG_SCHEMA, if_exists="append",
              index=False, method="multi", chunksize=1000)
    debug(f"  ✓ {dataset_name}: {len(df):,} rows -> {PG_SCHEMA}.{STAGING_TABLE}")
    return len(df)


def ensure_staging_table():
    """Create STAGING_TABLE with the fixed schema if it does not exist."""
    cols_sql = ",\n    ".join(
        f'"{c}" TIMESTAMP' if c == "date"
        else f'"{c}" DOUBLE PRECISION' if c == "value"
        else f'"{c}" TEXT'
        for c in STAGING_COLS
    )
    sql = f'CREATE TABLE IF NOT EXISTS "{PG_SCHEMA}".{STAGING_TABLE} (\n    {cols_sql}\n);'
    with engine.begin() as conn:
        conn.execute(text(sql))


def truncate_staging():
    """Empty STAGING_TABLE (keeps schema)."""
    with engine.begin() as conn:
        conn.execute(text(f'TRUNCATE TABLE "{PG_SCHEMA}".{STAGING_TABLE}'))
    debug(f"  ✓ truncated {PG_SCHEMA}.{STAGING_TABLE}")


REF_TIMESERIES_DDL = f"""
CREATE TABLE IF NOT EXISTS "{PG_SCHEMA}".{REF_TIMESERIES_TABLE} (
    source                  TEXT        NOT NULL,
    timeseries_id           TEXT        NOT NULL,
    timeseries_description  TEXT,
    source_dataset          TEXT,
    route                   TEXT,
    frequency               TEXT,
    duoarea                 TEXT,
    area_name               TEXT,
    product                 TEXT,
    product_name            TEXT,
    process                 TEXT,
    process_name            TEXT,
    units                   TEXT,
    published_date          TIMESTAMPTZ NOT NULL DEFAULT now(),
    PRIMARY KEY (source, timeseries_id)
);
"""

REF_TIMESERIES_UPSERT = f"""
INSERT INTO "{PG_SCHEMA}".{REF_TIMESERIES_TABLE} (
    source, timeseries_id, timeseries_description,
    source_dataset, route, frequency,
    duoarea, area_name,
    product, product_name,
    process, process_name,
    units
)
SELECT DISTINCT ON (series)
    'eia',
    series,
    "series-description",
    source_dataset,
    route,
    frequency,
    duoarea,
    "area-name",
    product,
    "product-name",
    process,
    "process-name",
    units
FROM "{PG_SCHEMA}".{{src}}
WHERE series IS NOT NULL
ORDER BY series, date DESC
ON CONFLICT (source, timeseries_id) DO NOTHING
"""


def ensure_ref_tables(regenerate=False):
    """Create ref_timeseries if missing. If regenerate=True, drop and recreate."""
    with engine.begin() as conn:
        if regenerate:
            debug(f"  ⚠ regenerate=True — dropping {REF_TIMESERIES_TABLE}")
            conn.execute(text(f'DROP TABLE IF EXISTS "{PG_SCHEMA}".{REF_TIMESERIES_TABLE} CASCADE'))
        conn.execute(text(REF_TIMESERIES_DDL))
        debug(f"    ✓ {PG_SCHEMA}.{REF_TIMESERIES_TABLE} ready")


def refresh_ref_timeseries_from(source_table):
    """Upsert distinct series metadata from `source_table` into ref_timeseries."""
    debug(f"  refreshing {REF_TIMESERIES_TABLE} from {PG_SCHEMA}.{source_table}...")
    try:
        with engine.begin() as conn:
            r = conn.execute(text(REF_TIMESERIES_UPSERT.format(src=source_table)))
            debug(f"    {REF_TIMESERIES_TABLE}: +{r.rowcount} new rows")
    except Exception as e:
        debug(f"    {REF_TIMESERIES_TABLE}: failed — {str(e)[:200]}")


debug("✓ functions ready: fetch_eia, stage_dataframe, ensure_staging_table, "
      "truncate_staging, ensure_ref_tables, refresh_ref_timeseries_from")

[2026-05-18 17:17:56] ✓ functions ready: fetch_eia, stage_dataframe, ensure_staging_table, truncate_staging, ensure_ref_tables, refresh_ref_timeseries_from


## 3. Create staging + ref + timeseries tables

Three tables ready to receive data. `timeseries` has FK to `ref_timeseries(source, timeseries_id)`, so Steps 4 → 5 → 6 must run in order (load staging, then catalogue, then facts).

In [4]:
ensure_staging_table()
ensure_ref_tables(regenerate=False)

# timeseries facts table — vintage-aware PK includes timeseries_published_date.
# FK enforces every loaded series exists in ref_timeseries.
TIMESERIES_DDL = f"""
CREATE TABLE IF NOT EXISTS "{PG_SCHEMA}".{TIMESERIES_TABLE} (
    source                    TEXT             NOT NULL,
    timeseries_id             TEXT             NOT NULL,
    timeseries_date           TIMESTAMP        NOT NULL,
    timeseries_value          DOUBLE PRECISION,
    timeseries_published_date TIMESTAMPTZ      NOT NULL DEFAULT now(),
    PRIMARY KEY (source, timeseries_id, timeseries_date, timeseries_published_date),
    CONSTRAINT {TIMESERIES_FK_NAME}
        FOREIGN KEY (source, timeseries_id)
        REFERENCES "{PG_SCHEMA}".{REF_TIMESERIES_TABLE} (source, timeseries_id)
);
"""

with engine.begin() as conn:
    conn.execute(text(TIMESERIES_DDL))

with engine.connect() as conn:
    n  = conn.execute(text(f'SELECT COUNT(*) FROM "{PG_SCHEMA}".{TIMESERIES_TABLE}')).scalar()
    fk = conn.execute(text(f"SELECT 1 FROM pg_constraint WHERE conname = '{TIMESERIES_FK_NAME}'")).scalar()

debug(f"✓ {PG_SCHEMA}.{TIMESERIES_TABLE} ready — {n:,} rows, FK={'yes' if fk else 'no'}")

[2026-05-18 17:17:56]     ✓ oil_network_data_loader.ref_timeseries ready
[2026-05-18 17:17:56] ✓ oil_network_data_loader.timeseries ready — 0 rows, FK=yes


## 4. Load EIA → `eia_staging`

Loop over all 27 EIA datasets. Each fetch is paginated; combined load is ~5–10 minutes depending on network speed. Empty fetches are skipped with a warning. HTTP errors are logged but don't halt the loop — bad endpoints get reported and we move on.

In [5]:
if RESET_STAGING:
    truncate_staging()

total_rows = 0
for ds in EIA_DATASETS:
    debug(f"▶ Fetching {ds['name']} ({ds['route']}, {ds['frequency']})")
    df = fetch_eia(route=ds["route"], facets=ds["facets"], frequency=ds["frequency"])
    debug(f"  shape: {df.shape}")
    total_rows += stage_dataframe(df, ds["name"], ds["route"], ds["frequency"])

debug(f"✓ staging load complete — {total_rows:,} rows in {PG_SCHEMA}.{STAGING_TABLE}")

with engine.connect() as conn:
    rows = conn.execute(text(
        f'SELECT source_dataset, COUNT(*) FROM "{PG_SCHEMA}".{STAGING_TABLE} '
        "GROUP BY source_dataset ORDER BY source_dataset"
    )).fetchall()
    debug("Row counts by source_dataset:")
    for name, n in rows:
        debug(f"  {name:<22} {n:>10,}")

[2026-05-18 17:17:56]   ✓ truncated oil_network_data_loader.eia_staging
[2026-05-18 17:17:56] ▶ Fetching production (petroleum/crd/crpdn, monthly)


[2026-05-18 17:17:59]     columns: ['period', 'duoarea', 'area-name', 'product', 'product-name', 'process', 'process-name', 'series', 'series-description', 'value', 'units']
[2026-05-18 17:17:59]     total  : 11,256


[2026-05-18 17:18:02]   shape: (11256, 11)


[2026-05-18 17:18:04]   ✓ production: 11,256 rows -> oil_network_data_loader.eia_staging
[2026-05-18 17:18:04] ▶ Fetching production_steo (steo, monthly)


[2026-05-18 17:18:05]     columns: ['period', 'seriesId', 'seriesDescription', 'value', 'unit']
[2026-05-18 17:18:05]     total  : 1,248
[2026-05-18 17:18:05]   shape: (1248, 5)


...

[2026-05-18 17:20:00]   shape: (13716, 11)


[2026-05-18 17:20:02]   ✓ supply_disposition: 13,716 rows -> oil_network_data_loader.eia_staging
[2026-05-18 17:20:02] ▶ Fetching trade_weekly (petroleum/move/wkly, weekly)


[2026-05-18 17:20:04]     columns: ['period', 'duoarea', 'area-name', 'product', 'product-name', 'process', 'process-name', 'series', 'series-description', 'value', 'units']
[2026-05-18 17:20:04]     total  : 6,523


[2026-05-18 17:20:06]   shape: (6523, 11)


[2026-05-18 17:20:07]   ✓ trade_weekly: 6,523 rows -> oil_network_data_loader.eia_staging
[2026-05-18 17:20:07] ✓ staging load complete — 161,329 rows in oil_network_data_loader.eia_staging
[2026-05-18 17:20:07] Row counts by source_dataset:
[2026-05-18 17:20:07]   crude_snd                   3,032
[2026-05-18 17:20:07]   exports                     6,250
[2026-05-18 17:20:07]   exports_padd                7,894
[2026-05-18 17:20:07]   exports_total               1,324
[2026-05-18 17:20:07]   imports                     6,952
[2026-05-18 17:20:07]   imports_entry               1,608
[2026-05-18 17:20:07]   imports_padd               15,094
[2026-05-18 17:20:07]   imports_processing          1,340
[2026-05-18 17:20:07]   movements_all               2,019
[2026-05-18 17:20:07]   movements_net               2,010
[2026-05-18 17:20:07]   movements_pipeline          1,300
[2026-05-18 17:20:07]   movements_rail              1,348
[2026-05-18 17:20:07]   movements_tanker              952
[202

## 5. Populate `ref_timeseries` from staging

One row per unique `series` across `eia_staging`. The `DISTINCT ON (series)` + `ORDER BY series, date DESC` pattern picks the most recent metadata row per series (in case the same series shows up multiple times with slightly different `area-name` strings, etc.). `ON CONFLICT DO NOTHING` keeps existing rows untouched on re-run.

In [6]:
refresh_ref_timeseries_from(STAGING_TABLE)

with engine.connect() as conn:
    n = conn.execute(text(
        f'SELECT COUNT(*) FROM "{PG_SCHEMA}".{REF_TIMESERIES_TABLE}'
    )).scalar()
    debug(f"  {REF_TIMESERIES_TABLE}: {n:,} unique series")

    debug("  Unique series per dataset:")
    rows = conn.execute(text(
        f'SELECT source_dataset, COUNT(*) FROM "{PG_SCHEMA}".{REF_TIMESERIES_TABLE} '
        "GROUP BY source_dataset ORDER BY source_dataset"
    )).fetchall()
    for name, c in rows:
        debug(f"    {name:<22} {c:>6,}")

sample = pd.read_sql(
    text(f'SELECT * FROM "{PG_SCHEMA}".{REF_TIMESERIES_TABLE} LIMIT 5'),
    engine,
)
display(sample)

[2026-05-18 17:20:07]   refreshing ref_timeseries from oil_network_data_loader.eia_staging...


[2026-05-18 17:20:09]     ref_timeseries: +1493 new rows
[2026-05-18 17:20:09]   ref_timeseries: 1,493 unique series
[2026-05-18 17:20:09]   Unique series per dataset:
[2026-05-18 17:20:09]     crude_snd                  19
[2026-05-18 17:20:09]     exports                   155
[2026-05-18 17:20:09]     exports_padd              245
[2026-05-18 17:20:09]     exports_total               6
[2026-05-18 17:20:09]     imports                   124
[2026-05-18 17:20:09]     imports_entry               9
[2026-05-18 17:20:09]     imports_padd              305
[2026-05-18 17:20:09]     imports_processing          7
[2026-05-18 17:20:09]     movements_all              18
[2026-05-18 17:20:09]     movements_net              14
[2026-05-18 17:20:09]     movements_pipeline         10
[2026-05-18 17:20:09]     movements_rail             18
[2026-05-18 17:20:09]     movements_tanker           10
[2026-05-18 17:20:09]     net_imports               112
[2026-05-18 17:20:09]     production            

,source,timeseries_id,timeseries_description,source_dataset,route,frequency,duoarea,area_name,product,product_name,process,process_name,units,published_date
0,eia,COPRAP,Crude Oil Production: Appalachia,production_steo,steo,monthly,None,None,None,None,None,None,million barrels per day,2026-05-18 16:20:07.771825+00:00
1,eia,COPRBK,Crude Oil Production: Bakken,production_steo,steo,monthly,None,None,None,None,None,None,million barrels per day,2026-05-18 16:20:07.771825+00:00
2,eia,COPREF,Crude Oil Production: Eagle Ford,production_steo,steo,monthly,None,None,None,None,None,None,million barrels per day,2026-05-18 16:20:07.771825+00:00
3,eia,COPRHA,Crude Oil Production: Haynesville,production_steo,steo,monthly,None,None,None,None,None,None,million barrels per day,2026-05-18 16:20:07.771825+00:00
4,eia,COPRPM,Crude Oil Production: Permian,production_steo,steo,monthly,None,None,None,None,None,None,million barrels per day,2026-05-18 16:20:07.771825+00:00


## 6. Populate `timeseries` from staging

Copies `(source='eia', series, date, value)` from `eia_staging` into the facts table. `timeseries_published_date` defaults to `now()`, so each run lands a new vintage. The PK `(source, timeseries_id, timeseries_date, timeseries_published_date)` means re-running the cell never overwrites an earlier vintage — it just appends. The FK to `ref_timeseries` would reject any series that didn't make it into the catalogue in Step 5.

`DISTINCT` handles the occasional exact-duplicate row that the EIA API returns. `ON CONFLICT DO NOTHING` is a belt-and-braces guard against duplicate (series, date) within a single run.

In [7]:
load_sql = f"""
INSERT INTO "{PG_SCHEMA}".{TIMESERIES_TABLE}
    (source, timeseries_id, timeseries_date, timeseries_value)
SELECT DISTINCT
    'eia', series, date, value
FROM "{PG_SCHEMA}".{STAGING_TABLE}
WHERE series IS NOT NULL
  AND date   IS NOT NULL
ON CONFLICT (source, timeseries_id, timeseries_date, timeseries_published_date) DO NOTHING
"""

with engine.begin() as conn:
    r = conn.execute(text(load_sql))
    debug(f"✓ {PG_SCHEMA}.{TIMESERIES_TABLE}: +{r.rowcount:,} rows inserted")

with engine.connect() as conn:
    n = conn.execute(text(f'SELECT COUNT(*) FROM "{PG_SCHEMA}".{TIMESERIES_TABLE}')).scalar()
    debug(f"  total rows: {n:,}")

    rows = conn.execute(text(f"""
        SELECT r.source_dataset, COUNT(*)
        FROM "{PG_SCHEMA}".{TIMESERIES_TABLE} t
        JOIN "{PG_SCHEMA}".{REF_TIMESERIES_TABLE} r
          ON t.source = r.source AND t.timeseries_id = r.timeseries_id
        GROUP BY r.source_dataset
        ORDER BY r.source_dataset
    """)).fetchall()
    debug("  Rows per dataset:")
    for ds, c in rows:
        debug(f"    {ds:<22} {c:>10,}")

[2026-05-18 17:20:15] ✓ oil_network_data_loader.timeseries: +146,089 rows inserted
[2026-05-18 17:20:15]   total rows: 146,089
[2026-05-18 17:20:15]   Rows per dataset:
[2026-05-18 17:20:15]     crude_snd                   2,228
[2026-05-18 17:20:15]     exports                     6,104
[2026-05-18 17:20:15]     exports_padd                7,212
[2026-05-18 17:20:15]     exports_total                 599
[2026-05-18 17:20:15]     imports                     6,668
[2026-05-18 17:20:15]     imports_entry               1,206
[2026-05-18 17:20:15]     imports_padd               14,077
[2026-05-18 17:20:15]     imports_processing            938
[2026-05-18 17:20:15]     movements_all               2,019
[2026-05-18 17:20:15]     movements_net               1,876
[2026-05-18 17:20:15]     movements_pipeline          1,300
[2026-05-18 17:20:15]     movements_rail              1,348
[2026-05-18 17:20:15]     movements_tanker              952
[2026-05-18 17:20:15]     net_imports              

## Verify

Quick summary: tables in the schema, total row counts, and the latest vintage's coverage span.

In [8]:
with engine.connect() as conn:
    tables = pd.read_sql(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = :s
        ORDER BY table_name
    """), conn, params={"s": PG_SCHEMA})

    counts = pd.read_sql(text(f"""
        SELECT 'eia_staging'    AS table_name, COUNT(*) AS rows FROM "{PG_SCHEMA}".{STAGING_TABLE}
        UNION ALL
        SELECT 'ref_timeseries' AS table_name, COUNT(*) AS rows FROM "{PG_SCHEMA}".{REF_TIMESERIES_TABLE}
        UNION ALL
        SELECT 'timeseries'     AS table_name, COUNT(*) AS rows FROM "{PG_SCHEMA}".{TIMESERIES_TABLE}
        ORDER BY table_name
    """), conn)

    coverage = pd.read_sql(text(f"""
        SELECT
            MIN(timeseries_date)            AS earliest_date,
            MAX(timeseries_date)            AS latest_date,
            COUNT(DISTINCT timeseries_id)   AS unique_series,
            COUNT(DISTINCT timeseries_published_date) AS vintages
        FROM "{PG_SCHEMA}".{TIMESERIES_TABLE}
    """), conn)

print(f"Tables in {PG_SCHEMA}:")
display(tables)
print("Row counts:")
display(counts)
print("Timeseries coverage:")
display(coverage)

Tables in oil_network_data_loader:


,table_name
0,eia_staging
1,ref_timeseries
2,timeseries


Row counts:


,table_name,rows
0,eia_staging,161329
1,ref_timeseries,1493
2,timeseries,146089


Timeseries coverage:


,earliest_date,latest_date,unique_series,vintages
0,2015-01-01,2027-12-01,1493,1
